# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _line item, meaning one product on one order._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


Explanation: I used json_normalize to split the lines list so each product gets its own row, and to copy the order and vendor_id onto every row, which gave me 4 rows adding up to 7 units.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

missing = joined[joined['_merge'] == 'left_only']
mismatched_vendor = missing['vendor_id'].unique()
units_vendor = missing['qty'].sum()

print('vendor that did not match:', mismatched_vendor)
print('units in vendor:', units_vendor)

joined

vendor that did not match: ['V-07']
units in vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


Explanation: I used a left join so every line item is kept even if the vendor has no name, and indicator=True added a _merge column that shows V-07 never matched with 3 units attached.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# I kept the unmatched vendor in the table and used its id as the name because dropping it would lose 3 units.
joined['name'] = joined['name'].fillna(joined['vendor_id'])

by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
vendor_total = by_vendor.sum()

print(by_vendor)
print('total:', vendor_total)

name
Hoos Burgers    3
V-07            3
Cav Merch       1
Name: qty, dtype: int64
total: 7


Explanation: I filled the blank vendor name with the vendor id so V-07 still shows up in the table, then grouped by name and added the quantities, which still comes out to 7 units.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)

zone_total = by_zone.sum()
all_units = joined['qty'].sum()

print(by_zone)
print('zone total:', zone_total)
print('all units:', all_units)

zone
A    4
Name: qty, dtype: int64
zone total: 4
all units: 7


Explanation: Zone A sold the most with 4 units, and it is the only zone that shows up. My zone total is 4 instead of 7 because groupby skips the V-07 row that has no zone, and that row has the missing 3 units.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Order 5 has no lines, so it has no line items and does not appear here.
rows = []
for order in ragged_json:
    for line in order.get('lines', []):
        rows.append({
            'order': order['order'],
            'vendor_id': order['vendor_id'],
            'item': line.get('item'),
            'qty': line.get('qty'),
        })

ragged = pd.DataFrame(rows)
missing = ragged['qty'].isna().sum()
zero = (ragged['qty'] == 0).sum()

print(ragged)
print('rows with a missing qty:', missing)
print('rows with a qty of zero:', zero)

   order vendor_id   item  qty
0      4      V-01   Soda  2.0
1      6      V-01  Fries  NaN
rows with a missing qty: 1
rows with a qty of zero: 0


Explanation: I used a loop with .get() so an order with no lines is skipped instead of crashing and a missing qty comes out as NaN. There is 1 row with a missing qty and 0 rows with a qty of zero.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is useful because the item names and quantities are in their own columns, so counting how many of each item sold can be done by just using a groupby(). If I had one row per order, all the items would be stuck in a list inside a single cell and it would be harder to access.

b) A zero means that nothing was sold, but a missing value means that we do not know what was sold. If I filled the missing one in with zero, the total would look complete when it is not and the average would be affected. Leaving it as NaN communicates that something is missing.